# 第 06 天：价值因子 1

> 来自《30 天因子研究计划》第 6 天  
> 主题：价值因子 1  
> 必做：PE / PB  
> 选做：EV / EBITDA  
> 目标产出：价值因子库

---

## 0. 今天你要真正学会什么？

前 5 天我们已经搭好了因子研究的检验框架：


因子值 → 未来收益标签 → IC → ICIR → 分层回测


从今天开始，我们进入具体因子库建设。第一个大类是价值因子。

价值因子的核心问题是：

> 这家公司相对于它的盈利、净资产、经营现金创造能力，到底贵不贵？

今天重点学习：

1. PE、PB、EV/EBITDA 分别是什么。
2. 为什么估值指标本身不是因子，必须先处理方向。
3. 为什么负利润、负净资产、极端值会让价值因子变脏。
4. 如何用 Python 构建一个最小价值因子库。
5. 如何把 PE、PB、EV/EBITDA 转成“越大越便宜”的因子分数。

一句话版：

> 价值因子不是简单买低 PE，而是把“便宜”这件事用可复现、可检验、可排序的方式表达出来。

---

## 1. 先建立直觉：买东西不只看价格

如果两家公司股价都是 10 元，哪只更便宜？

答案是不知道。

因为你还要知道：

- 每股赚多少钱？
- 每股净资产是多少？
- 公司有多少债务和现金？
- 经营利润能支撑多大的企业价值？

这就像买房：

- 房价 500 万不一定贵。
- 房价 100 万也不一定便宜。
- 你要看面积、地段、租金、贷款、现金流。

股票也是一样。

估值指标的目的，是把“价格”放到某个基本面尺度上比较。

---

## 2. 三个核心估值指标

### 2.1 PE：市盈率

PE 的公式：


PE = 总市值 / 净利润


也可以理解为：


PE = 每股价格 / 每股收益


直觉：

> 如果一家公司市值 100 亿、每年赚 10 亿，那么 PE = 10 倍。

常见解释：

- PE 低：相对于利润便宜。
- PE 高：相对于利润昂贵，或市场预期未来增长高。

风险点：

- 净利润为负时，PE 失去直观含义。
- 周期股利润高点时 PE 可能看起来很低，但反而危险。
- 一次性收益会扭曲净利润。

### 2.2 PB：市净率

PB 的公式：


PB = 总市值 / 净资产


直觉：

> 如果公司净资产 50 亿，市场给它 100 亿市值，那么 PB = 2 倍。

PB 更适合资产较重、净资产有意义的行业，比如银行、地产、周期制造。

风险点：

- 轻资产公司账面净资产可能很低，PB 会显得很高。
- 资产减值不足会让净资产虚高。
- 净资产为负时 PB 不可用。

### 2.3 EV/EBITDA：企业价值倍数

EV 是企业价值：


EV = 总市值 + 有息负债 - 现金


EBITDA 是息税折旧摊销前利润。


EV/EBITDA = 企业价值 / EBITDA


它比 PE 更关注经营层面的盈利能力，也会考虑债务和现金结构。

风险点：

- EBITDA 忽略资本开支。
- 对金融行业不一定适用。
- EBITDA 为负时不可用。

---

## 3. 因子方向：低估值如何变成高分数？

PE、PB、EV/EBITDA 都是：


越低越便宜


但很多因子研究希望统一成：


因子值越大越好


所以不能直接把 PE 当作正向价值因子。

常见做法：


pe_value = -log(PE)
pb_value = -log(PB)
ev_ebitda_value = -log(EV/EBITDA)


或者使用倒数：


EP = 净利润 / 总市值 = 1 / PE
BP = 净资产 / 总市值 = 1 / PB


第 7 天会重点讲 EP、BP 和股息率。今天先用 `-log(估值倍数)` 构建价值因子。

---

## 4. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260705)


如果缺包，可以先安装：


In [ ]:
pip install numpy pandas matplotlib


---

## 5. 构造模拟财务数据

真实研究中，你会从财务数据库读取总市值、净利润、净资产、债务、现金、EBITDA 等字段。今天用模拟数据讲清楚计算逻辑。


In [ ]:
n = 500

fundamental = pd.DataFrame({
    "ticker": [f"Stock_{i:03d}" for i in range(n)],
    "market_cap": rng.lognormal(mean=10.5, sigma=0.9, size=n),
    "net_income": rng.normal(loc=900, scale=500, size=n),
    "book_equity": rng.normal(loc=6000, scale=2500, size=n),
    "total_debt": rng.lognormal(mean=7.5, sigma=0.8, size=n),
    "cash": rng.lognormal(mean=7.0, sigma=0.9, size=n),
    "ebitda": rng.normal(loc=1300, scale=650, size=n),
})

# 人为制造一些异常情况：亏损、负净资产、负 EBITDA
fundamental.loc[rng.choice(n, size=35, replace=False), "net_income"] *= -1
fundamental.loc[rng.choice(n, size=15, replace=False), "book_equity"] *= -0.3
fundamental.loc[rng.choice(n, size=25, replace=False), "ebitda"] *= -1

fundamental.head()


字段含义：

| 字段 | 含义 |
| --- | --- |
| market_cap | 总市值 |
| net_income | 净利润 |
| book_equity | 净资产 |
| total_debt | 有息负债 |
| cash | 现金 |
| ebitda | EBITDA |

---

## 6. 计算 PE、PB、EV/EBITDA

### 6.1 基础计算


In [ ]:
fundamental["pe"] = fundamental["market_cap"] / fundamental["net_income"]
fundamental["pb"] = fundamental["market_cap"] / fundamental["book_equity"]
fundamental["enterprise_value"] = (
    fundamental["market_cap"] + fundamental["total_debt"] - fundamental["cash"]
)
fundamental["ev_ebitda"] = fundamental["enterprise_value"] / fundamental["ebitda"]

fundamental[["ticker", "pe", "pb", "enterprise_value", "ev_ebitda"]].head()


### 6.2 为什么要处理不可用值？

如果净利润为负，PE 可能是负数。  
负 PE 不是“超级便宜”，而是公司亏损。

如果净资产为负，PB 也失去常规意义。  
如果 EBITDA 为负，EV/EBITDA 也很难作为正常估值倍数。

所以我们先把不合理估值倍数设为空。


In [ ]:
for col in ["pe", "pb", "ev_ebitda"]:
    fundamental.loc[fundamental[col] <= 0, col] = np.nan

fundamental[["pe", "pb", "ev_ebitda"]].describe()


这一步看起来保守，但很重要。

> 在价值因子里，负估值倍数不是自动便宜，而通常是需要单独处理的异常样本。

---

## 7. 把估值倍数转成价值因子

### 7.1 简单工具函数


In [ ]:
def winsorize_series(s: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)


def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()


def value_from_multiple(multiple: pd.Series) -> pd.Series:
    """
    把估值倍数转成价值分数。
    倍数越低越便宜，价值分数越高。
    """
    x = np.log(multiple)
    x = winsorize_series(x)
    return -zscore(x)


### 7.2 构造三个价值因子


In [ ]:
fundamental["value_pe"] = value_from_multiple(fundamental["pe"])
fundamental["value_pb"] = value_from_multiple(fundamental["pb"])
fundamental["value_ev_ebitda"] = value_from_multiple(fundamental["ev_ebitda"])

factor_cols = ["value_pe", "value_pb", "value_ev_ebitda"]
fundamental[["ticker", "pe", "pb", "ev_ebitda"] + factor_cols].head()


解释：

- `value_pe` 越高，代表 PE 越低，越便宜。
- `value_pb` 越高，代表 PB 越低，越便宜。
- `value_ev_ebitda` 越高，代表 EV/EBITDA 越低，越便宜。

### 7.3 合成一个基础价值分数


In [ ]:
fundamental["value_score"] = fundamental[factor_cols].mean(axis=1, skipna=True)

fundamental[["ticker"] + factor_cols + ["value_score"]].head()


`mean(axis=1, skipna=True)` 的意思是：  
如果某只股票 PE 不可用，但 PB、EV/EBITDA 可用，仍然可以用剩余指标计算平均值。

---

## 8. 检查因子方向是否正确

我们希望：


估值倍数低 → 价值分数高


用相关性检查：


In [ ]:
direction_check = fundamental[["pe", "pb", "ev_ebitda", "value_pe", "value_pb", "value_ev_ebitda"]].corr()
direction_check.loc[["pe", "pb", "ev_ebitda"], ["value_pe", "value_pb", "value_ev_ebitda"]]


正常情况下：

- `pe` 和 `value_pe` 应该显著负相关。
- `pb` 和 `value_pb` 应该显著负相关。
- `ev_ebitda` 和 `value_ev_ebitda` 应该显著负相关。

---

## 9. 看看最便宜和最贵的股票


In [ ]:
cheap = fundamental.sort_values("value_score", ascending=False).head(10)
expensive = fundamental.sort_values("value_score", ascending=True).head(10)

cheap[["ticker", "pe", "pb", "ev_ebitda", "value_score"]]


In [ ]:
expensive[["ticker", "pe", "pb", "ev_ebitda", "value_score"]]


这个检查很朴素，但非常实用。

如果你发现最便宜的一批全是亏损公司、负净资产公司，说明处理逻辑有问题。

---

## 10. 价值因子和未来收益的模拟检验

为了把第 3-5 天知识串起来，我们人为生成一个未来收益标签，让便宜股票略微占优。


In [ ]:
noise = rng.normal(0, 0.06, size=n)
fundamental["future_20d_ret"] = 0.012 * fundamental["value_score"].fillna(0) + noise

ic = fundamental["value_score"].corr(fundamental["future_20d_ret"], method="spearman")
print("价值综合分数 Rank IC:", round(ic, 4))


按价值分数做 5 分组：


In [ ]:
valid = fundamental.dropna(subset=["value_score", "future_20d_ret"]).copy()
valid["group"] = pd.qcut(
    valid["value_score"].rank(method="first"),
    q=5,
    labels=["G1 最贵", "G2", "G3", "G4", "G5 最便宜"]
)

group_ret = valid.groupby("group", observed=True)["future_20d_ret"].mean()
group_ret


画图：


In [ ]:
group_ret.plot(kind="bar", title="价值分组未来 20 日平均收益")
plt.ylabel("Future 20D Return")
plt.xticks(rotation=30)
plt.show()


如果 G5 高于 G1，说明这个模拟世界里的价值因子是有效的。真实研究里当然需要更长时间、更严谨的检验。

---

## 11. 今日目标产出：价值因子库

我们把今天的逻辑封装成一个函数。


In [ ]:
def build_value_factor_library(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["pe"] = out["market_cap"] / out["net_income"]
    out["pb"] = out["market_cap"] / out["book_equity"]
    out["enterprise_value"] = out["market_cap"] + out["total_debt"] - out["cash"]
    out["ev_ebitda"] = out["enterprise_value"] / out["ebitda"]

    for col in ["pe", "pb", "ev_ebitda"]:
        out.loc[out[col] <= 0, col] = np.nan

    out["value_pe"] = value_from_multiple(out["pe"])
    out["value_pb"] = value_from_multiple(out["pb"])
    out["value_ev_ebitda"] = value_from_multiple(out["ev_ebitda"])
    out["value_score"] = out[["value_pe", "value_pb", "value_ev_ebitda"]].mean(axis=1, skipna=True)

    factor_library = out[[
        "ticker",
        "pe",
        "pb",
        "ev_ebitda",
        "value_pe",
        "value_pb",
        "value_ev_ebitda",
        "value_score",
    ]].copy()

    return factor_library


value_library = build_value_factor_library(fundamental)
value_library.head()


检查字段：


In [ ]:
value_library.describe()


这就是今天的目标产出：一个包含 PE、PB、EV/EBITDA 及其标准化价值分数的价值因子库。

---

## 12. 实战中要注意的细节

### 12.1 财务数据披露时点

不能在 1 月使用 3 月才披露的年报数据。

真实研究里，你需要使用“可得时点”数据，而不是财报期末数据。

### 12.2 行业差异

不同行业估值中枢差异很大。

例如：

- 银行 PB 常年偏低。
- 科技成长股 PE 常年偏高。
- 周期股利润波动导致 PE 容易误导。

所以价值因子后续常需要行业中性化。

### 12.3 负值和极端值

负利润、负净资产、极端估值倍数不能直接粗暴排序。

### 12.4 单指标不要神化

低 PE 可能是便宜，也可能是价值陷阱。  
低 PB 可能是低估，也可能是资产质量差。  
低 EV/EBITDA 可能是机会，也可能是高资本开支行业的假象。

---

## 13. 今天的知识图谱


In [ ]:
mindmap
  root((价值因子1))
    PE
      市值除以净利润
      低PE更便宜
      亏损时不可用
      周期利润会误导
    PB
      市值除以净资产
      低PB更便宜
      适合资产重行业
      负净资产不可用
    EV_EBITDA
      企业价值除以EBITDA
      考虑债务和现金
      EBITDA为负不可用
    因子处理
      log变换
      去极值
      标准化
      方向取负
    价值因子库
      value_pe
      value_pb
      value_ev_ebitda
      value_score
    风险点
      未来函数
      行业差异
      价值陷阱
      极端值


---

## 14. 初学者最容易踩的 8 个坑

### 坑 1：直接把 PE 当正向因子

PE 越低通常越便宜，所以如果你希望因子越大越好，要转换方向。

### 坑 2：把负 PE 当超级便宜

负 PE 通常代表亏损，不是估值低。

### 坑 3：忽略行业差异

银行和软件公司不能简单用同一个 PB 水平比较。

### 坑 4：只看一个估值指标

单个指标很容易失真。多个价值指标组合通常更稳。

### 坑 5：不做极端值处理

估值倍数很容易出现极端值，直接标准化会被少数样本带偏。

### 坑 6：忽略披露时点

这是基本面因子最常见的未来函数来源。

### 坑 7：把低估值等同于高收益

低估值可能是风险补偿，也可能是基本面恶化。

### 坑 8：不检查最便宜股票列表

因子构建后，一定要看头尾样本是否符合常识。

---

## 15. 今天的动手作业

### 作业 A：解释三个指标

用自己的话解释：

1. PE 是什么？
2. PB 是什么？
3. EV/EBITDA 相比 PE 多考虑了什么？

### 作业 B：运行价值因子库代码

运行本文所有 Python 代码，输出：

- `value_pe`
- `value_pb`
- `value_ev_ebitda`
- `value_score`

### 作业 C：检查因子方向

确认：


PE 越低，value_pe 越高
PB 越低，value_pb 越高
EV/EBITDA 越低，value_ev_ebitda 越高


### 作业 D：观察头尾股票

查看价值分数最高和最低的 10 只股票，判断是否符合常识。

### 作业 E：修改异常处理

尝试把 PE 大于 100 的样本设为空，再观察价值因子分布变化。

---

## 16. 自测题

### 题 1

PE 的公式是什么？

答案：总市值 / 净利润。

### 题 2

为什么负 PE 不应该直接当作便宜？

答案：负 PE 通常来自亏损，估值含义和正常盈利公司不同。

### 题 3

PB 更适合哪些行业？

答案：资产较重、净资产有意义的行业，比如银行、地产、部分周期制造。

### 题 4

为什么要对估值倍数取 `-log`？

答案：log 可以缓和极端倍数影响，负号让低估值转成高价值分数。

### 题 5

EV/EBITDA 中的 EV 包含什么？

答案：总市值 + 有息负债 - 现金。

---

## 17. 今日复盘模板


第 06 天复盘：价值因子 1

1. 我今天理解的 PE：

2. 我今天理解的 PB：

3. 我今天理解的 EV/EBITDA：

4. 我构建的价值因子字段：

5. 我检查出的异常值问题：

6. 我认为价值因子最容易踩的坑：

7. 明天学习 EP/BP/股息率前，我需要准备：


---

## 18. 明天预告：价值因子 2

明天会把 PE、PB 换一个方向来看：


EP = 净利润 / 市值
BP = 净资产 / 市值


还会加入股息率，形成更像报告的价值因子分析。

---

## 19. 一句话收尾

低估值不是魔法，价值因子也不是简单筛低 PE。

> 真正有用的价值因子，是把便宜、可比、可用、可检验这几件事同时做好。

---

## 20. 仅供学习的提醒

本文所有示例使用模拟数据，仅用于解释价值因子的计算方法，不构成任何投资建议。真实研究需要严格处理财务数据披露时点、复权市值、行业差异、异常值、交易成本和样本外验证。

---

# 统一高质量增强模块

> 本增强模块用于把第 06 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：价值因子1
- 必做：PE/PB
- 选做：EV/EBITDA
- 目标产出：价值因子库

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

低 PE 看起来便宜，但亏损公司的 PE 可能是负数；低 PB 看起来便宜，但资产质量可能很差。价值因子要先处理这些陷阱。

这个例子背后的关键直觉是：

> 便宜必须可比、可用、可检验。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


价值因子1
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 价值因子库


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(106)
n = 350
df = pd.DataFrame({
    "ticker": [f"S{i:03d}" for i in range(n)],
    "market_cap": rng.lognormal(10, 1, n),
    "net_income": rng.normal(800, 450, n),
    "book_equity": rng.normal(5000, 1600, n),
    "ebitda": rng.normal(1200, 500, n),
    "debt": rng.lognormal(7, .8, n),
    "cash": rng.lognormal(6.5, .7, n),
})
df["pe"] = df["market_cap"] / df["net_income"]
df["pb"] = df["market_cap"] / df["book_equity"]
df["ev_ebitda"] = (df["market_cap"] + df["debt"] - df["cash"]) / df["ebitda"]
for c in ["pe", "pb", "ev_ebitda"]:
    df.loc[df[c] <= 0, c] = np.nan

def zscore(s):
    return (s - s.mean()) / s.std()

def value_multiple(s):
    x = np.log(s).clip(np.log(s).quantile(.01), np.log(s).quantile(.99))
    return -zscore(x)

df["value_pe"] = value_multiple(df["pe"])
df["value_pb"] = value_multiple(df["pb"])
df["value_ev_ebitda"] = value_multiple(df["ev_ebitda"])
df["value_score"] = df[["value_pe", "value_pb", "value_ev_ebitda"]].mean(axis=1)
print(df[["value_pe", "value_pb", "value_ev_ebitda", "value_score"]].describe().round(3))


## E. 产出验收标准

完成今天课程后，你的 `价值因子库` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `价值因子1` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `价值因子库` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `价值因子库`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 06 天复盘：价值因子1

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
